In [5]:
import json
import os
import tempfile
from sklearn.metrics import accuracy_score, classification_report
import random
import requests
import statistics

# Filtering and sampling reviews

In [6]:
#this function filters for reviews that contain some given aspect score, set as aspect_score_key. needed bc score coverage varies within some directories like iclr_2017
def get_papers_with_score(aspect_score_key, search_dir):
    reviews_with_score = []
    search_dir_list = os.listdir(search_dir)
    for review_name in search_dir_list: 
        #opening each file in the directory to check if it contains the aspect score
        full_path = os.path.join(search_dir, review_name)
        with open(full_path, 'r') as f_open: 
            contents = json.load(f_open)
        contents_list =  contents.get("reviews")
        for review_dict in contents_list: 
            aspect_score = review_dict.get(aspect_score_key)
            if not (aspect_score == None): 
                #aspect score is present so append path
                reviews_with_score.append(full_path)
    return reviews_with_score

In [4]:
iclr_2017_train_reviews = "C:\\Users\\G34371231\\OneDrive - The George Washington University\\Desktop\\PeerRead\\data\\iclr_2017\\train\\reviews"
reviews_with_soundness = get_papers_with_score(aspect_score_key="SOUNDNESS_CORRECTNESS", search_dir = iclr_2017_train_reviews)

In [5]:
print(reviews_with_soundness)
print(len(reviews_with_soundness))

['C:\\Users\\G34371231\\OneDrive - The George Washington University\\Desktop\\PeerRead\\data\\iclr_2017\\train\\reviews\\304.json', 'C:\\Users\\G34371231\\OneDrive - The George Washington University\\Desktop\\PeerRead\\data\\iclr_2017\\train\\reviews\\304.json', 'C:\\Users\\G34371231\\OneDrive - The George Washington University\\Desktop\\PeerRead\\data\\iclr_2017\\train\\reviews\\304.json', 'C:\\Users\\G34371231\\OneDrive - The George Washington University\\Desktop\\PeerRead\\data\\iclr_2017\\train\\reviews\\304.json', 'C:\\Users\\G34371231\\OneDrive - The George Washington University\\Desktop\\PeerRead\\data\\iclr_2017\\train\\reviews\\305.json', 'C:\\Users\\G34371231\\OneDrive - The George Washington University\\Desktop\\PeerRead\\data\\iclr_2017\\train\\reviews\\305.json', 'C:\\Users\\G34371231\\OneDrive - The George Washington University\\Desktop\\PeerRead\\data\\iclr_2017\\train\\reviews\\305.json', 'C:\\Users\\G34371231\\OneDrive - The George Washington University\\Desktop\\PeerR

In [6]:
def sample_papers_with_score(paths_with_score, seed, size):
    #given a set seed and sample size, returns a random sample of reviews containing the aspect score. 
    random.seed(seed)
    return random.sample(paths_with_score, size)

In [7]:
sampled_review_paths = sample_papers_with_score(reviews_with_soundness, seed = 50, size = 100)
print(sampled_review_paths)

['C:\\Users\\G34371231\\OneDrive - The George Washington University\\Desktop\\PeerRead\\data\\iclr_2017\\train\\reviews\\397.json', 'C:\\Users\\G34371231\\OneDrive - The George Washington University\\Desktop\\PeerRead\\data\\iclr_2017\\train\\reviews\\357.json', 'C:\\Users\\G34371231\\OneDrive - The George Washington University\\Desktop\\PeerRead\\data\\iclr_2017\\train\\reviews\\379.json', 'C:\\Users\\G34371231\\OneDrive - The George Washington University\\Desktop\\PeerRead\\data\\iclr_2017\\train\\reviews\\422.json', 'C:\\Users\\G34371231\\OneDrive - The George Washington University\\Desktop\\PeerRead\\data\\iclr_2017\\train\\reviews\\353.json', 'C:\\Users\\G34371231\\OneDrive - The George Washington University\\Desktop\\PeerRead\\data\\iclr_2017\\train\\reviews\\395.json', 'C:\\Users\\G34371231\\OneDrive - The George Washington University\\Desktop\\PeerRead\\data\\iclr_2017\\train\\reviews\\372.json', 'C:\\Users\\G34371231\\OneDrive - The George Washington University\\Desktop\\PeerR

In [58]:
#get parsed_pdf paths based on sampled review paths
def get_parsed_pdf_paths(pdf_dir_path, sampled_review_paths):
    paper_paths = []
    for review_path in sampled_review_paths:
        paper_id = os.path.basename(review_path)
        paper_path = os.path.join(pdf_dir_path, os.path.splitext(paper_id)[0]) #need to add ".pdf.json"
        paper_path+=".pdf.json"
        paper_paths.append(paper_path)

    return paper_paths
    

In [60]:
pdf_dir_path = "C:\\Users\\G34371231\\OneDrive - The George Washington University\\Desktop\\PeerRead\\data\\iclr_2017\\train\\parsed_pdfs"
sampled_paper_paths = get_parsed_pdf_paths(pdf_dir_path, sampled_review_paths)

In [61]:
print(sampled_paper_paths)

['C:\\Users\\G34371231\\OneDrive - The George Washington University\\Desktop\\PeerRead\\data\\iclr_2017\\train\\parsed_pdfs\\397.pdf.json', 'C:\\Users\\G34371231\\OneDrive - The George Washington University\\Desktop\\PeerRead\\data\\iclr_2017\\train\\parsed_pdfs\\357.pdf.json', 'C:\\Users\\G34371231\\OneDrive - The George Washington University\\Desktop\\PeerRead\\data\\iclr_2017\\train\\parsed_pdfs\\379.pdf.json', 'C:\\Users\\G34371231\\OneDrive - The George Washington University\\Desktop\\PeerRead\\data\\iclr_2017\\train\\parsed_pdfs\\422.pdf.json', 'C:\\Users\\G34371231\\OneDrive - The George Washington University\\Desktop\\PeerRead\\data\\iclr_2017\\train\\parsed_pdfs\\353.pdf.json', 'C:\\Users\\G34371231\\OneDrive - The George Washington University\\Desktop\\PeerRead\\data\\iclr_2017\\train\\parsed_pdfs\\395.pdf.json', 'C:\\Users\\G34371231\\OneDrive - The George Washington University\\Desktop\\PeerRead\\data\\iclr_2017\\train\\parsed_pdfs\\372.pdf.json', 'C:\\Users\\G34371231\\One

# Model Prompting

In [22]:
#system prompt
soundness_guideline = f"""
  First, is the technical approach sound and well-chosen? 
  Second, can one trust the empirical claims of the paper -- are they supported by proper experiments and are the results of the experiments correctly interpreted?  
  5 = The approach is very apt, and the claims are convincingly supported. 
  4 = Generally solid work, although there are some aspects of the approach or evaluation I am not sure about. 
  3 = Fairly reasonable work. The approach is not bad, and at least the main claims are probably correct, but I am not entirely ready to accept them (based on the material in the paper). 
  2 = Troublesome. There are some ideas worth salvaging here, but the work should really have been done or evaluated differently. 
  1 = Fatally flawed.
  """

system_prompt = f"""You are a reviewer for a Natural Language Processing academic conference who critiques the soundness of papers based on the following guidelines:

soundness guidelines: {soundness_guideline}

Given a prompt containing paper contents, you respond with:
'score': The score you give the soundness of the paper according to the soundness guidelines. Your score is always an integer 1 to 5.
'critique': Your actionable critique of the soundness of the paper, which authors can immediately use to rewrite and improve their paper so it will be accepted to the conference. 
'headings': The headings of the paper for which section(s) should be rewritten to improve soundness.
Your critiques are critical, specific, and detailed. 
"""

In [23]:
def build_prompt(paper):
  metadata = paper.get('metadata') #metadata dictionary that contains the actual contents of the paper
  content_list = metadata.get('sections')
  #print(content_list)
  #print(content_list[4].get('text'))
  #print(type(content_list[4]))


  soundness_guideline = f"""
  First, is the technical approach sound and well-chosen? 
  Second, can one trust the empirical claims of the paper -- are they supported by proper experiments and are the results of the experiments correctly interpreted?  
  5 = The approach is very apt, and the claims are convincingly supported. 
  4 = Generally solid work, although there are some aspects of the approach or evaluation I am not sure about. 
  3 = Fairly reasonable work. The approach is not bad, and at least the main claims are probably correct, but I am not entirely ready to accept them (based on the material in the paper). 
  2 = Troublesome. There are some ideas worth salvaging here, but the work should really have been done or evaluated differently. 
  1 = Fatally flawed.
  """

  prompt_soundness = f""" You are a reviewer for an NLP academic conference and are critiquing this paper for its SOUNDNESS. Your task is to identify any issues and cite the 
  specific heading where the issue arises. 

  Paper Contents: {str(metadata.get('sections'))}

  Consider SOUNDNESS as folllows: {soundness_guideline} 

  Using the paper contents and the definition of SOUNDNESS, score the paper on SOUNDNESS, provide your justificaiton, and evidence with reference to headings.
  
  """
  #prompt = "give me three animals in the prediction, reasoning, evidence fields" did this to see if qwen would respond to something simple bc at first it returned nothing

  
   
  prompt_soundness_section_2 = f""" You are helping me write a submission for an NLP academic conference and are critiquing this paper for its SOUNDNESS. Your task is to identify any issues and cite the 
  specific heading where the issue arises. Be as critical and specific as possible.

  Paper Contents: {content_list[4].get('text')}

  SOUNDNESS is defined as folllows: {soundness_guideline} 

  Using the paper contents and the definition of SOUNDNESS, score the paper on SOUNDNESS (1-5), provide your justification, and references to .

  """


  prompt_soundness_3 = f""" You are critiquing this paper for its SOUNDNESS. Your task is to identify any issues and cite the 
  specific heading where the issue arises. Be as critical and specific as possible.

  Paper Contents: {content_list[4].get('text')}

  Consider SOUNDNESS as folllows: {soundness_guideline} 

  Using the paper contents and the definition of SOUNDNESS, score the paper on SOUNDNESS, provide your justification, and evidence with reference to headings.

  """
  
  prompt_soundness_actionable = f""" Based on the conference guidelines for scoring SOUNDNESS, you must score the provided paper contents on SOUNDNESS and provide an actionable critique. 

  SOUNDNESS scoring guidelines: {soundness_guideline} 

  Paper contents: {str(metadata.get('sections'))} 
  
  Based on the paper contents and the SOUNDNESS guidelines, score the SOUNDNESS of the paper (1-5), give your actionable critique, and cite the headings that need to be rewritten for better SOUNDNESS.
  """

#{content_list[4].get('text')}
 
  prompt_soundness_4 = f""" 

  Paper contents: {str(metadata.get('sections'))} 
  
  Based on the paper contents and the SOUNDNESS guidelines, score the SOUNDNESS of the paper (1-5), give your actionable critique, and cite the headings that need to be rewritten for better SOUNDNESS.
  """


  return prompt_soundness_actionable


In [24]:
pdf_path = "C:\\Users\\G34371231\\OneDrive - The George Washington University\\Desktop\\PeerRead\\data\\iclr_2017\\train\\parsed_pdfs\\304.pdf.json"
with open(pdf_path, 'r') as f1:
    paper = json.load(f1) #json file contents for one research paper

build_prompt(paper)

" Based on the conference guidelines for scoring SOUNDNESS, you must score the provided paper contents on SOUNDNESS and provide an actionable critique. \n\n  SOUNDNESS scoring guidelines: \n  First, is the technical approach sound and well-chosen? \n  Second, can one trust the empirical claims of the paper -- are they supported by proper experiments and are the results of the experiments correctly interpreted?  \n  5 = The approach is very apt, and the claims are convincingly supported. \n  4 = Generally solid work, although there are some aspects of the approach or evaluation I am not sure about. \n  3 = Fairly reasonable work. The approach is not bad, and at least the main claims are probably correct, but I am not entirely ready to accept them (based on the material in the paper). \n  2 = Troublesome. There are some ideas worth salvaging here, but the work should really have been done or evaluated differently. \n  1 = Fatally flawed.\n   \n\n  Paper contents: [{'heading': '1 INTRODUC

In [16]:
def model_forecasting(model, prompt):
    #print(prompt)
    # Send request to Ollama

    res = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": model, #llama3.2:3b , "qwen3:latest"
            "system": system_prompt,
            "prompt": prompt, 
            "stream": False, 
            #"think": True,
            # should i include format field?
            "format":{
            "type": "object",
            "properties":{ "score": {"type": "string"}, "critique": {"type":"string"}, "headings": {"type":"string"} }, 
            "required": ["score", "critique", "headings"]
            }
        }
    )
    result = res.json()
    return result


In [11]:
def predict_aspect(pdf_path, review_path, results):
    with open(pdf_path, 'r') as f1:
        paper = json.load(f1) #json file contents for one research paper

    prompt = build_prompt(paper)
    model = "llama3.2:latest"
    output = model_forecasting(model, prompt)
    print(output)
    json_response = json.loads(output.get("response"))

    results[paper.get("name")] = {
        "score": json_response.get("score"),
        "critique": json_response.get("critique"),
        "headings": json_response.get("headings")
    }
    print(results)
    return results

In [25]:
pdf_path = "C:\\Users\\G34371231\\OneDrive - The George Washington University\\Desktop\\PeerRead\\data\\acl_2017\\train\\parsed_pdfs\\699.pdf.json"
review_path = "C:\\Users\\G34371231\\OneDrive - The George Washington University\\Desktop\\PeerRead\\data\\acl_2017\\train\\reviews\\699.json"
results = {}
results = predict_aspect(pdf_path, review_path, results)

{'model': 'llama3.2:latest', 'created_at': '2025-07-15T15:46:34.5551375Z', 'response': '{\n\n"score": "While the paper presents a novel approach to keyphrase extraction using an encoder-decoder model with a copy mechanism, it suffers from several issues that impact its overall SOUNDNESS. The paper is well-written, and the experimental results are presented clearly. However, some methodological concerns and limitations need to be addressed for better SOUNDNESS. Here\'s a breakdown of the issues: 4/5"\n\n,"critique": "The authors present an innovative approach to keyphrase extraction using an encoder-decoder model with a copy mechanism. However, they need to explicitly discuss the challenges they faced during the development and training process. Moreover, they should provide more details about the experimental setup, including hyperparameter tuning and data preprocessing. Furthermore, they should evaluate their model against state-of-the-art methods for keyphrase extraction. Additionall

In [22]:
print(results)
output_path = "C:\\Users\\G34371231\\OneDrive - The George Washington University\\Desktop\\PeerRead\\dtais_summer\\aspect_review_1_section_prompt_actionable.json"
with open(output_path,'a') as f3:
    json.dump(results,f3)

{'699.pdf': {'score': 'She is a woman.', 'critique': 'She is a woman.', 'headings': 'She is a woman.'}}


In [ ]:
for paper in sampled_paper_paths: 
    results = predict_aspect()
    

# Measuring Accuracy

In [ ]:
def get_true_soundness(review_file_path): #returns the average of the soundness scores contained in the human generated reviews
    with open(review_file_path, 'r') as f_review:
        review_file_contents = json.load(f_review)
        
    review_list = review_file_contents.get("reviews")
    score_list = []
    for review_dict in review_list[1:]: 
        soundness = review_dict.get("SOUNDNESS_CORRECTNESS")
        
        if not (soundness == None): 
            #checking if soundness score is present bc not all reviews seem to have it
            score_list.append(int(soundness))
            
    if not score_list:
        print(f'SOUNDNESS_CORRECTNESS not found in {str(review_file_path)}')
        true_avg_score = None #file doesn't have this score
    else: 
        true_avg_score = statistics.mean(score_list)
       
    return true_avg_score

In [74]:
from sklearn.metrics import root_mean_squared_error

def get_soundness_accuracy(prediction_path, review_file_path):
    #how should i measure accuracy? percent error? st deviation? 
    #PeerRead paper uses root mean square error (not sure why not just MSE but RMSE?), OpenReviewer uses absolute difference
    
    true_scores = []
    predicted_scores = []

    with open(prediction_path, 'r') as f:
        prediction_dict = json.load(f)

    for paper in prediction_dict:
        response_dict = prediction_dict.get(paper)
        predicted_scores.append(response_dict.get("predicted"))
        review_file = str(os.path.splitext(paper)[0])+".json"
        review_full_path = os.path.join(review_file_path,review_file) #need to get full path 
        true_scores.append(get_true_soundness(review_full_path))


    print(true_scores)  
    print(predicted_scores) 
    rmse = root_mean_squared_error(true_scores, predicted_scores)
    print(rmse)
    


In [77]:
prediction_path = "C:\\Users\\G34371231\\OneDrive - The George Washington University\\Desktop\\PeerRead\\dtais_summer\\qwen3_soundness_paper_100_seed_50.json"
review_path = "C:\\Users\\G34371231\\OneDrive - The George Washington University\\Desktop\\PeerRead\\data\\acl_2017\\train\\reviews"
get_soundness_accuracy(prediction_path, review_path)


SOUNDNESS_CORRECTNESS not found in C:\Users\G34371231\OneDrive - The George Washington University\Desktop\PeerRead\data\acl_2017\train\reviews\759.json
SOUNDNESS_CORRECTNESS not found in C:\Users\G34371231\OneDrive - The George Washington University\Desktop\PeerRead\data\acl_2017\train\reviews\564.json
SOUNDNESS_CORRECTNESS not found in C:\Users\G34371231\OneDrive - The George Washington University\Desktop\PeerRead\data\acl_2017\train\reviews\627.json
SOUNDNESS_CORRECTNESS not found in C:\Users\G34371231\OneDrive - The George Washington University\Desktop\PeerRead\data\acl_2017\train\reviews\691.json
SOUNDNESS_CORRECTNESS not found in C:\Users\G34371231\OneDrive - The George Washington University\Desktop\PeerRead\data\acl_2017\train\reviews\343.json
SOUNDNESS_CORRECTNESS not found in C:\Users\G34371231\OneDrive - The George Washington University\Desktop\PeerRead\data\acl_2017\train\reviews\16.json
SOUNDNESS_CORRECTNESS not found in C:\Users\G34371231\OneDrive - The George Washington Un

ValueError: Input contains NaN.